In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_events_core (
    event_id STRING,
    event_type STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    actor_login STRING,
    repo_id BIGINT,
    repo_name STRING,
    org_id BIGINT,
    public BOOLEAN,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

existing_silver_ids = (
    spark.table(SILVER_CORE_TABLE)
    .select("event_id")
)

new_bronze_df = (
    bronze_df
    .join(
        existing_silver_ids,
        on="event_id",
        how="left_anti"
    )
)

print("New Bronze rows to transform:", new_bronze_df.count())

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

silver_core_df = (
    new_bronze_df
    .select(
        "event_id",
        "event_type",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("actor_json", "$.login")
            .alias("actor_login"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.get_json_object("repo_json", "$.name")
            .alias("repo_name"),

        F.get_json_object("org_json", "$.id")
            .cast("long")
            .alias("org_id"),

        "public",
        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
print("Rows:", silver_core_df.count())

silver_core_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("event_type").isNull().cast("int"))
        .alias("null_event_type"),

    F.sum(F.col("event_ts").isNull().cast("int"))
        .alias("null_event_ts"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id")
).show()

In [0]:
duplicate_count = (
    silver_core_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate event IDs:", duplicate_count)

In [0]:
silver_core_df.filter(
    F.col("repo_id").isNull()
).select(
    "event_id",
    "event_type",
    "event_ts",
    "actor_id",
    "actor_login",
    "repo_id",
    "repo_name",
    "source_file"
).show(truncate=False)

In [0]:
from delta.tables import DeltaTable

# Create table on first run
if not spark.catalog.tableExists(SILVER_CORE_TABLE):
    (
        silver_core_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_CORE_TABLE)
    )

silver_delta = DeltaTable.forName(spark, SILVER_CORE_TABLE)

(
    silver_delta.alias("t")
    .merge(
        silver_core_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver core MERGE completed.")

In [0]:
silver_count = spark.table(SILVER_CORE_TABLE).count()

duplicate_count = (
    spark.table(SILVER_CORE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Silver rows:", silver_count)
print("Duplicate event IDs:", duplicate_count)

In [0]:
event_tables = [
    "silver_push_events",
    "silver_pull_request_events",
    "silver_issue_events",
    "silver_issue_comment_events",
    "silver_watch_events",
    "silver_fork_events",
    "silver_release_events"
]

for table_name in event_tables:
    print(f"\n=== {table_name} ===")
    spark.sql(f"""
    DESCRIBE TABLE workspace.oss_ecosystem.{table_name}
    """).show(100, truncate=False)